# Coupling MPI Applications with PyTorch Inference and Training

**Estimated time:** ~45-60 minutes  
**Format:** 5 exercises. Each includes demo code, a small coding task, and a hidden solution.

---

## Session goals

This notebook builds a prototype coupled AI+HPC workflow using DragonHPC.
You will combine MPI-style application orchestration with PyTorch-based
inference and training.

You will practice how to:

- launch MPI applications with Dragon ProcessGroup
- capture selected rank output for downstream AI stages
- transform HPC outputs into model-ready tensors
- run PyTorch inference and training in cooperating processes
- compose an end-to-end coupled workflow

The exercises are written for beginners and intermediate users, while
providing extensible patterns for advanced users.

---

## Setup - run this first

Start Jupyter from a Dragon-enabled environment (for example with
`dragon-jupyter`) and run the next cell.

In [1]:
import os
import re
import queue
import socket
import time
from pathlib import Path

import dragon
import multiprocessing as mp
import dragon.native as dn
import torch

from dragon.infrastructure.facts import PMIBackend
from dragon.native.process import Process, ProcessTemplate, MSG_PIPE, MSG_DEVNULL
from dragon.native.process_group import ProcessGroup

try:
    mp.set_start_method("dragon")
except RuntimeError:
    pass

print("Dragon + PyTorch setup complete")
print("PyTorch version:", torch.__version__)

mpi_hello = Path("../dragon_native/mpi/mpi_hello").resolve()
print("mpi_hello exists:", mpi_hello.exists())

Dragon + PyTorch setup complete
PyTorch version: 2.13.0+cpu
mpi_hello exists: False


If `mpi_hello` is missing, build it once:

```bash
cd ../dragon_native/mpi
make
```

Then rerun Cell 2.

---

## Exercise 1 - Launch MPI and write tensor output to DDict

**Background:**

A common coupling pattern is to run an MPI application and put tensor data into
the DDict rather than scraping stdout. The producer here is a small mpi4py
program, [mpi_gen_data.py](mpi_gen_data.py), adapted from
[ai-in-the-loop/sim-expensive.c](ai-in-the-loop/sim-expensive.c): every rank
generates `(x, sin(x))` samples and each rank writes its own slice into the
DDict. A workflow *round* maps onto a DDict *checkpoint*, so each batch is
versioned and consumers can select exactly which round they want.

Demo pattern:

```python
pg = ProcessGroup(restart=False, pmi=PMIBackend.PMIX)
pg.add_process(
    nproc=total_ranks,
    template=ProcessTemplate(
        target=sys.executable,
        args=(str(exe_path), "--round", str(round_id), "--dser", ddict.serialize()),
        cwd=str(Path(exe_path).parent),
        stdout=MSG_DEVNULL,
    ),
)
pg.init(); pg.start(); pg.join(); pg.close()
```

Inside the MPI program each rank writes at the round's checkpoint:

```python
ddict.checkpoint_id = round_id
ddict[f"x_rank{rank}"] = x
ddict[f"y_rank{rank}"] = y
```

**Your task:**

1. Write `run_mpi_capture(exe_path, total_ranks, ddict, round_id=0)`.
2. Launch `total_ranks` MPI ranks that run `mpi_gen_data.py` with the serialized DDict.
3. Have each rank write its `(x, sin(x))` slice into the DDict at checkpoint `round_id`.
4. Ensure the group is joined and closed.


In [ ]:
# -- Exercise 1 -- your code here -----------------------------------------------
import sys
from pathlib import Path

from dragon.native.process import ProcessTemplate, MSG_DEVNULL
from dragon.native.process_group import ProcessGroup
from dragon.infrastructure.facts import PMIBackend


def run_mpi_capture(exe_path, total_ranks, ddict, round_id=0):
    """Launch the mpi4py generator so every rank writes its (x, sin(x)) slice
    into the shared DDict at checkpoint == round_id.

    This is the "producer" stage of the coupled workflow.  It adapts
    sim-expensive.c: instead of gathering to rank 0 and printing, each rank
    writes directly into the DDict.  The MPI ranks call ``ddict.checkpoint_id =
    round_id`` internally so this whole batch lands in a single versioned
    checkpoint.
    """
    exe_path = Path(exe_path).resolve()

    pg = ProcessGroup(restart=False, pmi=PMIBackend.PMIX)
    # A single template covers all ranks; the launcher is the Python
    # interpreter running our mpi4py program with the serialized DDict.
    pg.add_process(
        nproc=total_ranks,
        template=ProcessTemplate(
            target=sys.executable,
            args=(str(exe_path), "--round", str(round_id), "--dser", ddict.serialize()),
            cwd=str(exe_path.parent),
            stdout=MSG_DEVNULL,
        ),
    )

    pg.init()
    pg.start()
    pg.join()
    pg.close()


# Quick smoke test of the producer on its own.
MPI_SCRIPT = Path("mpi_gen_data.py").resolve()
if MPI_SCRIPT.exists():
    from dragon.data import DDict

    _dd = DDict(1, 1, 64 * 1024 * 1024, wait_for_keys=True, working_set_size=2)
    run_mpi_capture(MPI_SCRIPT, total_ranks=4, ddict=_dd, round_id=0)
    _dd.checkpoint_id = 0
    print("num_ranks written:", _dd["num_ranks"])
    print("rank 0 x sample:", _dd["x_rank0"][:3])
    _dd.destroy()
else:
    print("mpi_gen_data.py not found next to the notebook")


<details>
<summary><b>▶ Show Solution</b></summary>

```python
def run_mpi_capture(exe_path, total_ranks):
    pg = ProcessGroup(restart=False, pmi=PMIBackend.CRAY)
    pg.add_process(
        nproc=1,
        template=ProcessTemplate(target=exe_path, args=(), cwd=str(Path(exe_path).parent), stdout=MSG_PIPE),
    )
    if total_ranks > 1:
        pg.add_process(
            nproc=total_ranks - 1,
            template=ProcessTemplate(target=exe_path, args=(), cwd=str(Path(exe_path).parent), stdout=MSG_DEVNULL),
        )

    pg.init()
    pg.start()

    lines = []
    for puid in pg.puids:
        child = Process(None, ident=puid)
        if child.stdout_conn:
            try:
                while True:
                    lines.append(child.stdout_conn.recv().strip())
            except EOFError:
                pass

    pg.join()
    pg.close()
    return lines

exe = str(Path("../dragon_native/mpi/mpi_hello").resolve())
if Path(exe).exists():
    out = run_mpi_capture(exe, total_ranks=4)
    print("captured lines:", len(out))
    print(out[0] if out else "<no output>")
else:
    print("Build mpi_hello first in ../dragon_native/mpi")
```

</details>

---

## Exercise 2 - Aggregate tensors at the current checkpoint

Use `wait_for_keys` and `fetch_add` so a consumer only ever touches the keys of
a specific checkpoint.

**Background:**

Once the MPI producer has written a round, a consumer needs to gather every
rank's slice back into a single pair of tensors. The DDict gives us the
synchronization for free:

- `fetch_add("readers")` atomically hands the consumer a per-checkpoint reader
  id. Because the counter is a checkpointed key it is scoped to the round being
  read, letting the inference and trainer stages coordinate on the same
  versioned batch.
- `wait_for_keys=True` makes each read *block* until the producer has written
  that key at the selected checkpoint, so the consumer never sees a
  half-written round.

Demo pattern:

```python
ddict.checkpoint_id = round_id          # choose the round
num_ranks = ddict["num_ranks"]          # blocks until produced
x = torch.tensor(ddict[f"x_rank{r}"])   # blocks per rank until produced
```

**Your task:**

1. Write `get_data(ddict)` that reads at the DDict's current checkpoint.
2. Use `fetch_add` to claim a reader slot and `wait_for_keys` (via plain reads) to block until data is present.
3. Concatenate every rank's `x` and `y` slice and return the two tensors.


In [ ]:
# -- Exercise 2 -- your code here -----------------------------------------------
import torch


def get_data(ddict):
    """Aggregate every rank's tensor slice at the DDict's *current* checkpoint.

    Synchronization comes for free from the DDict:

    * ``fetch_add`` atomically hands this consumer a reader id for the current
      checkpoint.  Because it is a checkpointed key, the counter is scoped to
      the checkpoint we are reading, so independent consumers (inference and
      trainer) coordinate on the same versioned batch.
    * ``wait_for_keys=True`` makes each key read *block* until the MPI producer
      has written that rank's slice for this checkpoint.  A consumer therefore
      never sees a half-written round and only ever touches the keys belonging
      to the checkpoint it selected.

    The caller sets ``ddict.checkpoint_id`` before calling to choose the round.
    """
    reader_id = ddict.fetch_add("readers")  # unique, per-checkpoint reader slot

    num_ranks = ddict["num_ranks"]  # blocks until rank 0 posts it for this checkpoint

    xs, ys = [], []
    for r in range(num_ranks):
        # Each of these reads blocks until rank r has written THIS checkpoint.
        xs.append(torch.tensor(ddict[f"x_rank{r}"], dtype=torch.float32))
        ys.append(torch.tensor(ddict[f"y_rank{r}"], dtype=torch.float32))

    x = torch.cat(xs)
    y = torch.cat(ys)
    print(
        f"[get_data] reader {reader_id} read {num_ranks} ranks "
        f"at checkpoint {ddict.checkpoint_id} -> {tuple(x.shape)}",
        flush=True,
    )
    return x, y


<details>
<summary><b>▶ Show Solution</b></summary>

```python
def build_feature_tensor(lines):
    rows = []
    for line in lines:
        line_length = len(line)
        rank_count = line.lower().count("rank")
        digit_count = sum(ch.isdigit() for ch in line)
        rows.append([line_length, rank_count, digit_count])
    if not rows:
        return torch.zeros((0, 3), dtype=torch.float32)
    return torch.tensor(rows, dtype=torch.float32)

sample = [
    "Hello, world, rank 0 on host x",
    "rank 1 finished step 12",
    "no rank keyword here",
]
x = build_feature_tensor(sample)
print(x)
print("shape:", tuple(x.shape))
```

</details>

---

## Exercise 3 - PyTorch inference service process

**Background:**

A lightweight inference process can consume the MPI-produced batches and return
a quality score each round. Here the model (a linear layer over polynomial
features, adapted from [ai-in-the-loop/model.py](ai-in-the-loop/model.py))
approximates `sin(x)`. The service should compute the **max norm** (infinity
norm) error between its prediction and the MPI `sin(x)` data and return that
value on a queue. This mirrors online scoring in coupled workflows.

Demo pattern:

```python
model.load_state_dict(ddict["model_weights"])   # latest weights from trainer
with torch.no_grad():
    pred = model(make_features(x)).squeeze(-1)
max_norm_error = torch.linalg.vector_norm(pred - y, ord=float("inf")).item()
out_q.put(max_norm_error)
```

**Your task:**

1. Write `inference_worker(ddict, out_q, event, total_rounds)`.
2. Loop over rounds, checking `event` each iteration and stopping early once it is set.
3. For each round, select the checkpoint, `get_data`, load `model_weights`, and run a forward pass under `torch.no_grad()`.
4. Put the max-norm error onto `out_q` for the convergence checker.


In [ ]:
# -- Exercise 3 -- your code here -----------------------------------------------
import torch

# --- Tiny model, adapted from ai-in-the-loop/model.py --------------------------
# We approximate sin(x) with a linear model over polynomial features
# [1, x, x^2, ..., x^(BASIS_DEGREE-1)].  No scipy needed for the poly basis.
BASIS_DEGREE = 6


def make_features(x):
    """Build polynomial features of shape [N, BASIS_DEGREE] from a 1-D tensor."""
    x = x.unsqueeze(1)  # [N, 1]
    feats = [x**i for i in range(BASIS_DEGREE)]
    return torch.cat(feats, dim=1).to(torch.float32)


def build_model():
    """A single linear layer mapping polynomial features -> scalar."""
    torch.manual_seed(0)
    return torch.nn.Linear(BASIS_DEGREE, 1)


def inference_worker(ddict, out_q, event, total_rounds):
    """Score the current model against each round of MPI-produced data.

    For every round the worker:
      1. checks the shared ``event`` and stops early once convergence is
         signaled,
      2. selects that round's checkpoint and blocks (via wait_for_keys) until
         ``get_data`` returns the batch,
      3. loads the latest weights the trainer published under the persistent
         ``model_weights`` key,
      4. computes the max-norm (infinity-norm) error between the prediction and
         the MPI ``sin(x)`` targets and pushes it onto ``out_q`` for the
         convergence checker.
    """
    model = build_model()

    round_id = 0
    while not event.is_set() and round_id < total_rounds:
        ddict.checkpoint_id = round_id
        x, y = get_data(ddict)  # blocks until this round has been produced

        # Persistent key: always present after the parent seeds it, and updated
        # asynchronously by the trainer.
        model.load_state_dict(ddict["model_weights"])
        model.eval()
        with torch.no_grad():
            pred = model(make_features(x)).squeeze(-1)

        max_norm_error = torch.linalg.vector_norm(pred - y, ord=float("inf")).item()
        print(f"[infer] round {round_id} max-norm error = {max_norm_error:.3e}", flush=True)
        out_q.put(max_norm_error)

        round_id += 1


<details>
<summary><b>▶ Show Solution</b></summary>

```python
def inference_worker(in_q, out_q):
    model = torch.nn.Sequential(
        torch.nn.Linear(3, 8),
        torch.nn.ReLU(),
        torch.nn.Linear(8, 1),
    )
    model.eval()

    while True:
        batch = in_q.get()
        if batch is None:
            break
        with torch.no_grad():
            pred = model(batch)
        out_q.put(pred)

in_q = mp.Queue()
out_q = mp.Queue()
worker = dn.Process(target=inference_worker, args=(in_q, out_q))
worker.start()

x = torch.tensor([[10.0, 1.0, 2.0], [20.0, 0.0, 4.0]], dtype=torch.float32)
in_q.put(x)
print(out_q.get())

in_q.put(None)
worker.join()
```

</details>

---

## Exercise 4 - PyTorch training on streamed HPC batches

**Background:**

Training runs as a separate stage that consumes the MPI batches as they land in
the DDict. Because the DDict keeps a working set of checkpoints alive, the
trainer can fit on the last *n* most recent checkpoints at once. When it
finishes a round it republishes its weights under the **persistent** key
`model_weights` (via `pput`) so the inference worker immediately scores the
newer model.

Demo pattern:

```python
for c in range(max(0, round_id - history + 1), round_id + 1):
    ddict.checkpoint_id = c
    x, y = get_data(ddict)            # gather recent-history batches
...
loss = torch.nn.functional.smooth_l1_loss(model(feats), target)
loss.backward(); optimizer.step()
ddict.pput("model_weights", model.state_dict())   # persistent, cross-checkpoint
```

**Your task:**

1. Write `trainer_worker(ddict, event, total_rounds, history=3)`.
2. Loop over rounds, checking `event` each iteration and stopping early once it is set.
3. Each round, gather batches from the last `history` checkpoints, run a few optimizer steps.
4. Publish the updated weights to the DDict at the persistent key `model_weights`.


In [ ]:
# -- Exercise 4 -- your code here -----------------------------------------------
import torch


def trainer_worker(ddict, event, total_rounds, history=3):
    """Train on the most recent ``history`` checkpoints, round after round.

    Because the DDict keeps a working set of checkpoints alive, the trainer can
    reach back over the last few rounds of MPI data and fit on all of them at
    once.  Each round it:
      1. checks the shared ``event`` and stops early on convergence,
      2. gathers batches from the last ``history`` checkpoints (each read blocks
         until that round's data exists),
      3. runs a few optimizer steps, and
      4. republishes the updated weights under the persistent ``model_weights``
         key so the inference worker immediately scores the newer model.
    """
    model = build_model()
    model.load_state_dict(ddict["model_weights"])
    optimizer = torch.optim.Adam(model.parameters(), lr=0.05)

    round_id = 0
    while not event.is_set() and round_id < total_rounds:
        # Collect training data from the last `history` checkpoints.
        xs, ys = [], []
        for c in range(max(0, round_id - history + 1), round_id + 1):
            ddict.checkpoint_id = c
            x, y = get_data(ddict)  # blocks until checkpoint c is available
            xs.append(x)
            ys.append(y)
        x = torch.cat(xs)
        target = torch.cat(ys).unsqueeze(-1)
        feats = make_features(x)

        model.train()
        for _ in range(50):
            optimizer.zero_grad()
            loss = torch.nn.functional.smooth_l1_loss(model(feats), target)
            loss.backward()
            optimizer.step()

        # Persistent (checkpoint-independent) publish so every consumer,
        # regardless of the checkpoint it is on, reads the freshest weights.
        ddict.pput("model_weights", model.state_dict())
        print(f"[train] round {round_id} loss = {loss.item():.3e}", flush=True)

        round_id += 1


<details>
<summary><b>▶ Show Solution</b></summary>

```python
def trainer_worker(train_q, result_q):
    model = torch.nn.Sequential(
        torch.nn.Linear(3, 16),
        torch.nn.ReLU(),
        torch.nn.Linear(16, 1),
    )
    opt = torch.optim.Adam(model.parameters(), lr=1e-2)

    final_loss = None
    while True:
        item = train_q.get()
        if item is None:
            break
        x_batch, y_batch = item
        pred = model(x_batch)
        loss = torch.nn.functional.mse_loss(pred, y_batch)
        opt.zero_grad()
        loss.backward()
        opt.step()
        final_loss = float(loss.item())

    result_q.put(final_loss if final_loss is not None else float("nan"))

train_q = mp.Queue()
result_q = mp.Queue()
trainer = dn.Process(target=trainer_worker, args=(train_q, result_q))
trainer.start()

for i in range(5):
    x = torch.randn(8, 3)
    y = (0.3 * x[:, :1] + 0.1 * x[:, 1:2] - 0.2 * x[:, 2:3])
    train_q.put((x, y))

train_q.put(None)
trainer.join()
print("final loss:", result_q.get())
```

</details>

---

## Exercise 5 - Build a prototype coupled AI+HPC workflow

**Background:**

This capstone couples the MPI producer with the PyTorch inference and training
workers **and** a convergence checker into one asynchronous mini workflow. The
goal is clean stage boundaries and robust process orchestration where the DDict
is used both as the data-exchange medium and as the synchronization primitive.

Every worker checks a shared `Event` as part of its loop. The convergence
checker reads the inference errors off the result queue and **sets the event
once 3 consecutive results are below `1e-6`**, which halts the whole pipeline.

Demo architecture:

```text
parent ─ round loop ─► run_mpi_capture ──► DDict (checkpoint == round)
                                            │        ▲
                 ┌──────────────────────────┼────────┘  wait_for_keys blocks
                 ▼                           ▼           consumers until data
         inference_worker ──► result Queue ──► convergence_checker
                 │                                  │ sets Event after 3
         trainer_worker ◄── model_weights (pput) ◄──┘ consecutive errors < 1e-6
```

**Your task:**

1. Create a DDict with `wait_for_keys=True` and a working set big enough to keep every round alive; seed `model_weights` with `pput`.
2. Start `inference_worker`, `trainer_worker`, and a `convergence_checker` as Dragon processes sharing an `Event` and a result `Queue`.
3. Write the `convergence_checker` so it sets the `Event` after 3 consecutive sub-`1e-6` errors.
4. Drive the producer round loop, then join every stage and destroy the DDict.


In [ ]:
# -- Exercise 5 -- Prototype coupled AI+HPC workflow ----------------------------
# Architecture (all stages run concurrently as Dragon processes):
#
#   parent ─ round loop ─► run_mpi_capture ──► DDict (checkpoint == round)
#                                               │        ▲
#                    ┌──────────────────────────┼────────┘  wait_for_keys blocks
#                    ▼                           ▼           consumers until data
#            inference_worker ──► result Queue ──► convergence_checker
#                    │                                  │ sets Event after 3
#            trainer_worker ◄── model_weights (pput) ◄──┘ consecutive errors < 1e-6
#                    │
#            every worker checks the Event each loop iteration and exits early
#
import sys
from pathlib import Path

from dragon.data import DDict
from dragon.native.event import Event
from dragon.native.queue import Queue
import dragon.native as dn

MPI_SCRIPT = Path("mpi_gen_data.py").resolve()
CONVERGENCE_TOL = 1e-6
CONVERGENCE_PATIENCE = 3


def convergence_checker(result_q, event, tol, patience, total_rounds):
    """Watch the stream of inference errors and signal convergence.

    Reads one max-norm error per round from ``result_q``.  When ``patience``
    (=3) consecutive errors fall below ``tol`` (=1e-6) it sets the shared
    ``event`` so every worker in the pipeline halts on its next loop check.
    """
    consecutive = 0
    for _ in range(total_rounds):
        error = result_q.get()  # blocks for the next inference result
        consecutive = consecutive + 1 if error < tol else 0
        print(
            f"[converge] error={error:.3e} consecutive(<{tol:g})={consecutive}",
            flush=True,
        )
        if consecutive >= patience:
            print("[converge] convergence reached -> setting event", flush=True)
            event.set()
            return
    print("[converge] round budget exhausted without convergence", flush=True)


def run_coupled_workflow(exe_path, total_ranks=4, total_rounds=8):
    # Working set large enough to keep every round's checkpoint alive so the
    # trainer can look back over recent history without checkpoints retiring.
    ddict = DDict(
        1,
        1,
        256 * 1024 * 1024,
        wait_for_keys=True,
        working_set_size=total_rounds + 2,
    )

    # Seed the persistent weights so both consumers can read from round 0.
    ddict.pput("model_weights", build_model().state_dict())

    event = Event()
    result_q = Queue()

    inferer = dn.Process(target=inference_worker, args=(ddict, result_q, event, total_rounds))
    trainer = dn.Process(target=trainer_worker, args=(ddict, event, total_rounds))
    checker = dn.Process(
        target=convergence_checker,
        args=(result_q, event, CONVERGENCE_TOL, CONVERGENCE_PATIENCE, total_rounds),
    )

    inferer.start()
    trainer.start()
    checker.start()

    # Producer stage: generate one MPI batch per round.  ``run_mpi_capture``
    # blocks until the round's job finishes, so the fast consumers spend most of
    # their time parked in ``get_data`` waiting on the next checkpoint.  When
    # convergence is signaled we write ONE more checkpoint before stopping so
    # any consumer already blocked on the next round unblocks, sees the event on
    # its next loop check, and exits cleanly (no deadlock on an unwritten key).
    for r in range(total_rounds):
        run_mpi_capture(exe_path, total_ranks, ddict, round_id=r)
        if event.is_set():
            if r + 1 < total_rounds:
                run_mpi_capture(exe_path, total_ranks, ddict, round_id=r + 1)
            print(f"[producer] convergence signaled; stopping after round {r}", flush=True)
            break

    inferer.join()
    trainer.join()
    checker.join()

    print("Workflow summary")
    print("  converged:", event.is_set())
    print("  rounds requested:", total_rounds)
    ddict.destroy()


if MPI_SCRIPT.exists():
    run_coupled_workflow(MPI_SCRIPT, total_ranks=4, total_rounds=8)
else:
    print("Generate mpi_gen_data.py next to the notebook first")


<details>
<summary><b>▶ Show Solution</b></summary>

```python
def run_coupled_workflow(exe_path):
    lines = run_mpi_capture(exe_path, total_ranks=4)
    x = build_feature_tensor(lines)

    if x.shape[0] == 0:
        print("No MPI lines captured; aborting workflow")
        return

    y = (0.05 * x[:, :1] + 0.01 * x[:, 1:2] + 0.02 * x[:, 2:3])

    train_q = mp.Queue()
    train_result_q = mp.Queue()
    infer_in_q = mp.Queue()
    infer_out_q = mp.Queue()

    trainer = dn.Process(target=trainer_worker, args=(train_q, train_result_q))
    inferer = dn.Process(target=inference_worker, args=(infer_in_q, infer_out_q))

    trainer.start()
    inferer.start()

    train_q.put((x, y))
    train_q.put(None)

    infer_in_q.put(x)
    pred = infer_out_q.get()
    infer_in_q.put(None)

    trainer.join()
    inferer.join()

    final_loss = train_result_q.get()
    print("Workflow summary")
    print("  captured lines:", len(lines))
    print("  final training loss:", final_loss)
    print("  prediction shape:", tuple(pred.shape))

exe = str(Path("../dragon_native/mpi/mpi_hello").resolve())
if Path(exe).exists():
    run_coupled_workflow(exe)
else:
    print("Build mpi_hello first in ../dragon_native/mpi")
```

</details>

---

## Bonus Exercise - Agents-in-the-loop 

**Background:**

This capstone exercise couples an MPI producer with PyTorch training and inference workers and a convergence checker in one mini workflow. The goal is an asynchronous workflow with robust process orchestration.

Demo architecture:

In [ ]:
# -- Bonus Exercise -- Agents-in-the-loop -----------------------------------------------
!export OMP_PROC_BIND='close'
!export OMP_PLACES='cores'
import lightweight_agent_workflow as workflow
# The ultralight workflow is recommended if your hardware did not support running the model in Course 2 or it was very slow for you.
#import ultralight_agent_workflow as workflow

init_cfls = [0.3, 2.4, 9.9, 0.8]
num_ranks = 4
iterations = 2
user_prompt = (
        "Check the latest CFD result arrays for NaNs and report which "
        "CFL values were stable and which produced NaNs. "
        "Choose the next set of CFL values, increasing CFL for "
        "stable ranks and decreasing it for ranks that produced "
        "NaNs."
)
start = time.perf_counter()
workflow.run(init_cfls, num_ranks, iterations, user_prompt)
print(f"runtime: {time.perf_counter() - start}",flush=True)

[startup] Launching lightweight inference service...
[startup] Agent 'generator_agent' ready.
[startup] Agent 'nan_agent' ready.

[fn] run_experiments_node -> cfl_values=[0.3, 2.4, 9.9, 0.8]
Got a job <dragon.workflows.batch.batch.Job object at 0xffff64f97140>
Got a job <dragon.workflows.batch.batch.Job object at 0xffff64f975f0>
Got a job <dragon.workflows.batch.batch.Job object at 0xffff64f0c0e0>
Got a job <dragon.workflows.batch.batch.Job object at 0xffff6611c200>
Got ecodes: [0, 0, 0, 0]
[inference] Loading model from /workspaces/pearc_agenticloop_clean/model on cpu...
Got ecodes: [0, 0, 0, 0]


Loading weights:   0%|          | 0/326 [00:00<?, ?it/s]

Got ecodes: [0, 0, 0, 0]


Loading weights:  19%|█▊        | 61/326 [00:11<00:36,  7.17it/s]

Got ecodes: [0, 0, 0, 0]
Iteration 1/2
Request: Check the latest CFD result arrays for NaNs and report which CFL values were stable and which produced NaNs. Choose the next set of CFL values, increasing CFL for stable ranks and decreasing it for ranks that produced NaNs.



Loading weights: 100%|██████████| 326/326 [00:57<00:00,  5.67it/s]


[inference] Model ready — serving requests.



[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


[inference] payload='{"response": {"type": "tool_request", "tool_calls": [{"name": "scan_all_ranks", "args": {}}]}}'
[tool] Result for key: cfl_0.3_rank0: {'has_nans': np.False_, 'max_value': np.float64(1.0), 'min_value': np.float64(0.0)}
[tool] Result for key: cfl_0.3_rank1: {'has_nans': np.False_, 'max_value': np.float64(1.0), 'min_value': np.float64(-2.9826216189002254e-60)}
[tool] Result for key: cfl_0.3_rank2: {'has_nans': np.False_, 'max_value': np.float64(0.0), 'min_value': np.float64(0.0)}
[tool] Result for key: cfl_0.3_rank3: {'has_nans': np.False_, 'max_value': np.float64(0.0), 'min_value': np.float64(0.0)}
[tool] Result for key: cfl_2.4_rank0: {'has_nans': np.True_, 'max_value': np.float64(5.402089789225298e+23), 'min_value': np.float64(-5.2466050888169935e+23)}
[tool] Result for key: cfl_9.9_rank0: {'has_nans': np.True_, 'max_value': np.float64(12539710463388.508), 'min_value': np.float64(-11748928330917.176)}
[tool] Result for key: cfl_0.8_rank0: {'has_nans': np.False_, 'm

---

## Summary

You built a prototype coupled AI+HPC pipeline using DragonHPC ProcessGroup
and PyTorch stages for both inference and training.

| Concept | API |
|---|---|
| MPI launch orchestration | `ProcessGroup(..., pmi=PMIBackend.CRAY)` |
| Rank output capture | `MSG_PIPE` + `stdout_conn.recv()` |
| Quiet non-head ranks | `MSG_DEVNULL` |
| Feature conversion | Python parsing -> `torch.tensor` |
| Inference service | separate process + queues + `torch.no_grad()` |
| Training stage | separate process + optimizer + loss backward |
| Workflow composition | MPI producer + AI workers + parent coordinator |

### Next steps

- Replace toy features with domain-specific simulation outputs.
- Swap toy models for your production PyTorch model checkpoints.
- Add multi-node placement policies and telemetry for throughput tuning.